In [19]:
import os
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import PropertyGraphIndex, Document, Settings
# Volvemos a traer a Gemini para el trabajo sucio
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.ollama import OllamaEmbedding

print("⚙️ Configurando Motor Híbrido: Gemini (Extractor) + Ollama (Vectores)...")

# Aseguramos la API Key
if "GEMINI_API_KEY" in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

# 1. Gemini será el Cerebro
llm = Gemini(model="models/gemini-2.5-flash", api_key=os.environ.get("GOOGLE_API_KEY"))
# 2. Ollama sigue siendo el motor de búsqueda vectorial
embed_model = OllamaEmbedding(model_name="nomic-embed-text", base_url="http://localhost:11434")

Settings.llm = llm
Settings.embed_model = embed_model

texto_mini = """
Ebenezer Scrooge era un viejo avaro que odiaba la Navidad. Trabajaba en su despacho en Londres, 
junto a su empleado Bob Cratchit, a quien pagaba una miseria. Cratchit tiritaba de frío porque 
Scrooge no le dejaba encender la estufa.

Una Nochebuena, Scrooge recibió la visita del fantasma de su antiguo socio, Jacob Marley. 
Marley estaba condenado a arrastrar pesadas cadenas forjadas por su avaricia en vida, y le 
advirtió a Scrooge que le pasaría lo mismo si no cambiaba de actitud.
"""
documentos = [Document(text=texto_mini)]

print("🧠 Construyendo el Property Graph con Gemini...")
# Forzamos la reconstrucción desde cero
index = PropertyGraphIndex.from_documents(documentos, show_progress=True)

elementos_extraidos = index.property_graph_store.get()
# ¡Fíjate en este número cuando se ejecute!
print(f"📊 RESULTADO DE LA EXTRACCIÓN: {len(elementos_extraidos)} elementos extraídos.")

print("\n" + "="*50 + "\n")

pregunta = "¿Quién es Marley, qué le pasa y qué le dice a Scrooge?"
print(f"❓ Pregunta: {pregunta}")

query_engine = index.as_query_engine(include_text=True)
respuesta = await query_engine.aquery(pregunta)

print("\n💡 RESPUESTA DE LLAMAINDEX:")
print(respuesta.response)

⚙️ Configurando Motor Híbrido: Gemini (Extractor) + Ollama (Vectores)...


/var/folders/tf/rh2grj_j239fbyx5vlzz54r40000gn/T/ipykernel_8619/2834242057.py:17: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  llm = Gemini(model="models/gemini-2.5-flash", api_key=os.environ.get("GOOGLE_API_KEY"))


🧠 Construyendo el Property Graph con Gemini...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 2/2 [00:00<00:00,  7.40it/s]


📊 RESULTADO DE LA EXTRACCIÓN: 14 elementos extraídos.


❓ Pregunta: ¿Quién es Marley, qué le pasa y qué le dice a Scrooge?

💡 RESPUESTA DE LLAMAINDEX:
Marley es el antiguo socio de Scrooge. Está condenado a arrastrar pesadas cadenas forjadas por su avaricia en vida. Le advierte a Scrooge que le pasará lo mismo si no cambia de actitud.


In [23]:
# Primero, inspeccionemos qué hay en los elementos
elementos = index.property_graph_store.get()

print(f"Total elementos: {len(elementos)}")
print("\n--- Tipos de elementos ---")
for i, item in enumerate(elementos[:5]):  # solo los primeros 5
    print(f"\nElemento {i}: {type(item).__name__}")
    print(f"  Atributos: {[a for a in dir(item) if not a.startswith('_')]}")
    print(f"  Repr: {item}")

Total elementos: 14

--- Tipos de elementos ---

Elemento 0: ChunkNode
  Atributos: ['construct', 'copy', 'dict', 'embedding', 'from_orm', 'id', 'id_', 'json', 'label', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'parse_file', 'parse_obj', 'parse_raw', 'properties', 'schema', 'schema_json', 'text', 'update_forward_refs', 'validate']
  Repr: Ebenezer Scrooge era un viejo avaro que odiaba la Navidad. Trabajaba en su despacho en Londres, 
junto a su empleado Bob Cratchit, a quien pagaba una miseria. Cratchit tiritaba de frío porque 
Scrooge no le dejaba encender la estufa.

Una Nochebuena, Scrooge recibió la visita del fantasma de su antiguo socio, Jacob Marley. 
Marley estaba condenado a arrastrar pesadas cadenas forjadas por s

In [24]:
from llama_index.core.graph_stores.types import Relation, EntityNode, ChunkNode

elementos = index.property_graph_store.get()

# Separar por tipo
entidades = [e for e in elementos if isinstance(e, EntityNode)]
relaciones = [e for e in elementos if isinstance(e, Relation)]
chunks = [e for e in elementos if isinstance(e, ChunkNode)]

print(f"EntityNodes: {len(entidades)}")
print(f"Relations: {len(relaciones)}")
print(f"ChunkNodes: {len(chunks)}")

# Inspeccionar una relación si existe
if relaciones:
    r = relaciones[0]
    print(f"\nEjemplo de Relation: {r}")
    print(f"Atributos: {[a for a in dir(r) if not a.startswith('_')]}")
else:
    print("\n⚠️ No hay Relations, probando get_rel_map...")
    # Alternativa: obtener relaciones directamente
    for entidad in entidades[:3]:
        rels = index.property_graph_store.get_rel_map(
            graph_nodes=[entidad], depth=1
        )
        print(f"\nRelaciones de '{entidad.name}': {rels}")

EntityNodes: 13
Relations: 0
ChunkNodes: 1

⚠️ No hay Relations, probando get_rel_map...

Relaciones de 'Ebenezer scrooge': [(EntityNode(label='entity', embedding=None, properties={'triplet_source_id': '1166a9cf-ca29-4fab-ac8a-1e9ed53b8c4e'}, name='Ebenezer scrooge'), Relation(label='Pagaba miseria a', source_id='Ebenezer scrooge', target_id='Bob cratchit', properties={'triplet_source_id': '1166a9cf-ca29-4fab-ac8a-1e9ed53b8c4e'}), EntityNode(label='entity', embedding=None, properties={'triplet_source_id': '1166a9cf-ca29-4fab-ac8a-1e9ed53b8c4e'}, name='Bob cratchit')), (EntityNode(label='entity', embedding=None, properties={'triplet_source_id': '1166a9cf-ca29-4fab-ac8a-1e9ed53b8c4e'}, name='Bob cratchit'), Relation(label='Era empleado de', source_id='Bob cratchit', target_id='Ebenezer scrooge', properties={'triplet_source_id': '1166a9cf-ca29-4fab-ac8a-1e9ed53b8c4e'}), EntityNode(label='entity', embedding=None, properties={'triplet_source_id': '1166a9cf-ca29-4fab-ac8a-1e9ed53b8c4e'}, nam

In [28]:
from pyvis.network import Network
from llama_index.core.graph_stores.types import EntityNode, Relation

print("🎨 Construyendo el mapa interactivo del Grafo...")

net = Network(directed=True, height="700px", width="100%", bgcolor="#1a1a1a", font_color="white", cdn_resources='remote')
net.repulsion(node_distance=200, central_gravity=0.2, spring_length=150)

relaciones_vistas = set()
relaciones_dibujadas = 0

for entidad in entidades:
    triplets = index.property_graph_store.get_rel_map(
        graph_nodes=[entidad], depth=1
    )
    for (nodo_origen, relacion, nodo_destino) in triplets:
        # Saltar si alguno de los nodos es ChunkNode (no tiene .name)
        if not isinstance(nodo_origen, EntityNode) or not isinstance(nodo_destino, EntityNode):
            continue
        
        origen = nodo_origen.name
        destino = nodo_destino.name
        etiqueta = relacion.label
        
        clave = (origen, etiqueta, destino)
        if clave in relaciones_vistas:
            continue
        relaciones_vistas.add(clave)
        
        net.add_node(origen, label=origen, color="#00ffcc", shape="dot", size=25)
        net.add_node(destino, label=destino, color="#ff5555", shape="dot", size=25)
        net.add_edge(origen, destino, title=etiqueta, label=etiqueta, color="#aaaaaa")
        relaciones_dibujadas += 1

print(f"✅ Se han dibujado {relaciones_dibujadas} flechas de conexión.")

archivo_html = "grafo_scrooge_llamaindex_final.html"
net.write_html(archivo_html)
print(f"🎉 Archivo '{archivo_html}' generado.")

🎨 Construyendo el mapa interactivo del Grafo...
✅ Se han dibujado 10 flechas de conexión.
🎉 Archivo 'grafo_scrooge_llamaindex_final.html' generado.
